In [1]:
from pathlib import Path
import pandas as pd

# VS Code runs a notebook with its own folder as the working directory,
# so the raw data is one level up from notebooks/.
DATA_RAW = Path("..") / "data" / "raw"

# Expected files (exact names from the Kaggle download).
EXPECTED_FILES = {
    "customers":      "olist_customers_dataset.csv",
    "geolocation":    "olist_geolocation_dataset.csv",
    "order_items":    "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews":  "olist_order_reviews_dataset.csv",
    "orders":         "olist_orders_dataset.csv",
    "products":       "olist_products_dataset.csv",
    "sellers":        "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Fail early with a clear message if anything is missing.
missing = [f for f in EXPECTED_FILES.values() if not (DATA_RAW / f).exists()]
assert not missing, f"Missing files in {DATA_RAW.resolve()}: {missing}"
print("All 9 files found in", DATA_RAW.resolve())

All 9 files found in /workspaces/olist-logistics-intelligence/data/raw


In [2]:
# Load each CSV into a dictionary of DataFrames: tables["orders"], etc.
# No cleaning here: we want to see the data exactly as delivered.
tables = {name: pd.read_csv(DATA_RAW / file) for name, file in EXPECTED_FILES.items()}
print("Loaded", len(tables), "tables")

Loaded 9 tables


In [3]:
# Summarise the size of each table.
summary = pd.DataFrame(
    {
        "rows": {name: df.shape[0] for name, df in tables.items()},
        "columns": {name: df.shape[1] for name, df in tables.items()},
    }
)
summary

,rows,columns
customers,99441,5
geolocation,1000163,5
order_items,112650,7
order_payments,103886,5
order_reviews,99224,7
orders,99441,8
products,32951,9
sellers,3095,4
category_translation,71,2


In [4]:
# Column names and types for every table, so we work from what the data says.
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print(df.dtypes.to_string())


=== customers ===
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str

=== geolocation ===
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str

=== order_items ===
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64

=== order_payments ===
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64

=== order_reviews ===
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_mes

In [11]:
def check_key(df, cols):
    """Report whether `cols` behave like a primary key: no nulls, no duplicates."""
    cols = [cols] if isinstance(cols, str) else list(cols)
    return {
        "rows": len(df),
        "distinct_keys": len(df.drop_duplicates(subset=cols)),
        "rows_with_null_key": int(df[cols].isna().any(axis=1).sum()),
        "rows_in_duplicated_keys": int(df.duplicated(subset=cols, keep=False).sum()),
    }

# Candidate keys from Step 1.1 (reviews are tested with two candidates).
candidate_keys = {
    "customers":            "customer_id",
    "orders":               "order_id",
    "order_items":          ["order_id", "order_item_id"],
    "order_payments":       ["order_id", "payment_sequential"],
    "order_reviews (review_id)":            "review_id",
    "order_reviews (review_id + order_id)": ["review_id", "order_id"],
    "products":             "product_id",
    "sellers":              "seller_id",
    "category_translation": "product_category_name",
}

key_results = {}
for label, cols in candidate_keys.items():
    table_name = label.split(" ")[0]
    key_results[label] = check_key(tables[table_name], cols)

key_report = pd.DataFrame(key_results).T
key_report["is_valid_pk"] = (
    (key_report["rows"] == key_report["distinct_keys"])
    & (key_report["rows_with_null_key"] == 0)
)
key_report

,rows,distinct_keys,rows_with_null_key,rows_in_duplicated_keys,is_valid_pk
customers,99441,99441,0,0,True
orders,99441,99441,0,0,True
order_items,112650,112650,0,0,True
order_payments,103886,103886,0,0,True
order_reviews (review_id),99224,98410,0,1603,False
order_reviews (review_id + order_id),99224,99224,0,0,True
products,32951,32951,0,0,True
sellers,3095,3095,0,0,True
category_translation,71,71,0,0,True


In [6]:
customers = tables["customers"]
orders = tables["orders"]

print("customer rows:               ", len(customers))
print("distinct customer_id:        ", customers["customer_id"].nunique())
print("distinct customer_unique_id: ", customers["customer_unique_id"].nunique())

# Attach the real-person ID to each order. validate= makes pandas raise an
# error if the right-hand key is not unique, so a bad join cannot slip through.
orders_with_person = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)
print("\norders before join:", len(orders))
print("orders after join: ", len(orders_with_person))

# How many orders does each real person have?
orders_per_person = orders_with_person.groupby("customer_unique_id").size()
print("\nOrders per unique customer:")
print(orders_per_person.value_counts().sort_index().to_string())

customer rows:                99441
distinct customer_id:         99441
distinct customer_unique_id:  96096

orders before join: 99441
orders after join:  99441

Orders per unique customer:
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1


In [7]:
reviews = tables["order_reviews"]

# 1) Is review_id repeated across rows?
dup_review_rows = reviews.duplicated(subset="review_id", keep=False).sum()
print("rows sharing a review_id with another row:", dup_review_rows)

# 2) How many reviews does each order have?
reviews_per_order = reviews.groupby("order_id").size()
print("\nReviews per order:")
print(reviews_per_order.value_counts().sort_index().to_string())

# 3) Orders in the orders table with no review at all.
orders_without_review = (~orders["order_id"].isin(reviews["order_id"])).sum()
print("\norders with no review:", orders_without_review)

rows sharing a review_id with another row: 1603

Reviews per order:
1    98126
2      543
3        4

orders with no review: 768


In [8]:
# Only show columns that actually contain nulls.
for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"{name}: no missing values\n")
    else:
        report = pd.DataFrame({
            "missing": nulls,
            "pct_missing": (nulls / len(df) * 100).round(2),
        })
        print(f"{name}:\n{report.to_string()}\n")

customers: no missing values

geolocation: no missing values

order_items: no missing values

order_payments: no missing values

order_reviews:
                        missing  pct_missing
review_comment_title      87656        88.34
review_comment_message    58247        58.70

orders:
                               missing  pct_missing
order_approved_at                  160         0.16
order_delivered_carrier_date      1783         1.79
order_delivered_customer_date     2965         2.98

products:
                            missing  pct_missing
product_category_name           610         1.85
product_name_lenght             610         1.85
product_description_lenght      610         1.85
product_photos_qty              610         1.85
product_weight_g                  2         0.01
product_length_cm                 2         0.01
product_height_cm                 2         0.01
product_width_cm                  2         0.01

sellers: no missing values

category_translation: n

In [9]:
def count_orphans(child, child_col, parent, parent_col):
    """Child rows whose key has no match in the parent table."""
    return int((~child[child_col].isin(parent[parent_col])).sum())

fk_checks = [
    ("orders.customer_id",            "orders",         "customer_id", "customers", "customer_id"),
    ("order_items.order_id",          "order_items",    "order_id",    "orders",    "order_id"),
    ("order_items.product_id",        "order_items",    "product_id",  "products",  "product_id"),
    ("order_items.seller_id",         "order_items",    "seller_id",   "sellers",   "seller_id"),
    ("order_payments.order_id",       "order_payments", "order_id",    "orders",    "order_id"),
    ("order_reviews.order_id",        "order_reviews",  "order_id",    "orders",    "order_id"),
    ("products.product_category_name","products",       "product_category_name",
                                                         "category_translation", "product_category_name"),
]

rows = []
for label, child, ccol, parent, pcol in fk_checks:
    rows.append({"foreign_key": label,
                 "orphan_rows": count_orphans(tables[child], ccol, tables[parent], pcol)})
pd.DataFrame(rows)

# The reverse direction: parents with no children.
print("orders with no items:   ", int((~orders["order_id"].isin(tables["order_items"]["order_id"])).sum()))
print("orders with no payment: ", int((~orders["order_id"].isin(tables["order_payments"]["order_id"])).sum()))

orders with no items:    775
orders with no payment:  1


In [10]:
items = tables["order_items"]
payments = tables["order_payments"]

# --- Correct total: sum each table separately at its own grain ---
payments_total = payments["payment_value"].sum()
print(f"Total payment_value (payments table alone): {payments_total:,.2f}")

# --- Wrong way: join orders to BOTH child tables at once ---
naive = orders.merge(items, on="order_id", how="left").merge(payments, on="order_id", how="left")
print(f"\norders rows:           {len(orders):,}")
print(f"after joining items:   {len(orders.merge(items, on='order_id', how='left')):,}")
print(f"after also payments:   {len(naive):,}")
print(f"Naive total payment_value: {naive['payment_value'].sum():,.2f}")

# --- Right way: collapse the payments table to one row per order first ---
payments_per_order = payments.groupby("order_id", as_index=False)["payment_value"].sum()
safe = orders.merge(payments_per_order, on="order_id", how="left", validate="one_to_one")
print(f"\nSafe join rows: {len(safe):,}")
print(f"Safe total payment_value: {safe['payment_value'].sum():,.2f}")

Total payment_value (payments table alone): 16,008,872.12

orders rows:           99,441
after joining items:   113,425
after also payments:   118,434
Naive total payment_value: 20,470,726.66

Safe join rows: 99,441
Safe total payment_value: 16,008,872.12


In [12]:
print(pd.DataFrame(rows).to_string())

                      foreign_key  orphan_rows
0              orders.customer_id            0
1            order_items.order_id            0
2          order_items.product_id            0
3           order_items.seller_id            0
4         order_payments.order_id            0
5          order_reviews.order_id            0
6  products.product_category_name          623


In [13]:
print(orders["order_status"].value_counts().to_string())

no_items = ~orders["order_id"].isin(tables["order_items"]["order_id"])
print("\nStatus of orders with no items:")
print(orders.loc[no_items, "order_status"].value_counts().to_string())

delivered = orders["order_status"] == "delivered"
print("\nDelivered orders missing a customer delivery date:",
      int(orders.loc[delivered, "order_delivered_customer_date"].isna().sum()))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Status of orders with no items:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1

Delivered orders missing a customer delivery date: 8


In [14]:
# Which 13 products have a category that is not in the translation table?
prod = tables["products"]
trans = tables["category_translation"]
unmatched = prod[prod["product_category_name"].notna()
                 & ~prod["product_category_name"].isin(trans["product_category_name"])]
print(unmatched["product_category_name"].value_counts().to_string())

# Are the 610 null-category products the same rows missing name/description/photos?
four = ["product_category_name", "product_name_lenght",
        "product_description_lenght", "product_photos_qty"]
print("\nRows missing all four:", int(prod[four].isna().all(axis=1).sum()))
print("Rows missing at least one:", int(prod[four].isna().any(axis=1).sum()))

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3

Rows missing all four: 610
Rows missing at least one: 610
